# Shalini Speakathon — prepare the missing 9 videos

**For Days 12–14 and 16–21.** This notebook signs into *your* Google Drive and compresses each original MP4 to less than 240 MiB without cropping it. The nine outputs are saved in a folder named **Shalini Missing Speakathon MP4s (compressed)** in your Drive.

Choose **Runtime → Run all**, sign in to the Google account with access to your source folder, and wait until **FINISHED** appears. Some files are large, so the run may take a while. Your originals remain unchanged. This is temporary preparation; the website will host the files locally, without Drive embeds.


In [ ]:
# Shalini Speakathon — compress missing challenge videos directly in YOUR Google Drive.
# The original 9 MP4s are each above ChatGPT's 256 MiB connector download limit.
# Run this cell in Google Colab and select the Google account that can view the challenge folder.
import os, re, io, json, time, shutil, subprocess
from pathlib import Path
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
svc = build("drive", "v3", credentials=creds, cache_discovery=False)

ROOT_ID = "1UCQwp_kUaU-Bxnjdv2d7yIcT8YQWlVSJ"
DEST_NAME = "Shalini Missing Speakathon MP4s (compressed)"
NEEDED = [12, 13, 14, 16, 17, 18, 19, 20, 21]
WORK = Path("/content/shalini_speakathon_temp")
WORK.mkdir(parents=True,exist_ok=True)
TARGET_BYTES = 185 * 1024 * 1024
MAX_BYTES = 240 * 1024 * 1024

def list_all(q, fields="nextPageToken,files(id,name,mimeType,size)", page_size=1000):
    items, page = [], None
    while True:
        answer = svc.files().list(q=q, fields=fields,pageSize=page_size,pageToken=page,supportsAllDrives=True,includeItemsFromAllDrives=True).execute()
        items.extend(answer.get("files",[]))
        page = answer.get("nextPageToken")
        if not page: return items

def shell(args):
    process = subprocess.run(args, capture_output=True,text=True)
    if process.returncode:
        raise RuntimeError("FFmpeg failed: "+process.stderr[-1600:])
    return process.stdout

def duration_of(path):
    raw=shell(["ffprobe","-v","error","-show_entries","format=duration","-of","json",str(path)])
    duration=float(json.loads(raw)["format"]["duration"])
    if duration<=0: raise RuntimeError("Missing video duration")
    return duration

def download_file(file_id,path):
    print(" Downloading to Colab temporary storage...")
    request=svc.files().get_media(fileId=file_id)
    with path.open("wb") as stream:
        downloader=MediaIoBaseDownload(stream,request,chunksize=16*1024*1024)
        done=False
        last=0
        while not done:
            status,done=downloader.next_chunk(num_retries=5)
            current=round(status.progress()*100)
            if current>=last+15 or done:
                print(f"  Download {current}%")
                last=current

def compress(input_path,output_path):
    seconds=duration_of(input_path)
    # Set an average video bitrate to keep EACH output well below 256 MiB.
    budget_kbps=TARGET_BYTES*8/seconds/1000
    vkbps=max(130,min(1600,int(budget_kbps*0.86-90)))
    for attempt in range(4):
        print(f" Encoding 9:16 source at {vkbps} kbps (attempt {attempt+1})...")
        shell(["ffmpeg","-nostdin","-y","-hide_banner","-loglevel","error",
               "-i",str(input_path),"-map","0:v:0","-map","0:a:0?",
               "-vf","scale='min(540,iw)':-2,fps=25",
               "-c:v","libx264","-preset","veryfast",
               "-b:v",f"{vkbps}k","-maxrate",f"{int(vkbps*1.2)}k","-bufsize",f"{int(vkbps*2)}k",
               "-pix_fmt","yuv420p","-c:a","aac","-b:a","64k",
               "-movflags","+faststart",str(output_path)])
        size=output_path.stat().st_size
        print(f"  Compressed file: {size/1048576:.1f} MiB")
        if size<MAX_BYTES and size>100000: return size
        vkbps=max(110,int(vkbps*.68))
    raise RuntimeError("Unable to make MP4 smaller than 240 MiB")

def upload(path,folder_id):
    media=MediaFileUpload(str(path), mimetype="video/mp4",resumable=True,chunksize=8*1024*1024)
    request=svc.files().create(body={"name":path.name,"parents":[folder_id],"mimeType":"video/mp4"},
                              media_body=media,fields="id,name,size",supportsAllDrives=True)
    response=None
    while response is None:
        status,response=request.next_chunk(num_retries=5)
    return response

folders=list_all(f"'{ROOT_ID}' in parents and trashed = false and mimeType = 'application/vnd.google-apps.folder'")
folder_for_day={}
for item in folders:
    m=re.fullmatch(r"Challenge\s+(\d+)",item["name"].strip(),flags=re.I)
    if m:folder_for_day[int(m.group(1))]=item["id"]

destinations=list_all("name = '"+DEST_NAME+"' and trashed = false and mimeType = 'application/vnd.google-apps.folder'")
if destinations:
    dest_id=destinations[0]["id"]
else:
    dest_id=svc.files().create(body={"name":DEST_NAME,"mimeType":"application/vnd.google-apps.folder"},
                              fields="id").execute()["id"]
print("Destination:",f"https://drive.google.com/drive/folders/{dest_id}")
print("Original folders accessible:",len(folder_for_day),"; remaining challenge days:",NEEDED)

successes,failures=[],[]
for day in NEEDED:
    output_name=f"day-{day:02d}.mp4"
    try:
        existing=list_all(f"'{dest_id}' in parents and name = '{output_name}' and trashed = false")
        if existing and int(existing[0].get("size",0))>100000:
            print(f"\nDay {day}: already completed, skipping duplicate.")
            successes.append(day);continue
        folder_id=folder_for_day.get(day)
        if not folder_id:raise RuntimeError(f"No access to the Challenge {day} folder; sign in with the correct Google account")
        candidates=list_all(f"'{folder_id}' in parents and trashed = false")
        video_files=[f for f in candidates if f.get("mimeType","").startswith("video/") or f["name"].lower().endswith(".mp4")]
        if not video_files:raise RuntimeError("No MP4 found in challenge folder")
        video=video_files[0]
        raw_path=WORK/f"source-day-{day:02d}.mp4"
        out_path=WORK/output_name
        print(f"\n===== DAY {day} · {video['name']} · {int(video.get('size',0))/1048576:.1f} MiB =====")
        download_file(video["id"],raw_path)
        compress(raw_path,out_path)
        uploaded=upload(out_path,dest_id)
        print("  Uploaded:",uploaded["name"], f"https://drive.google.com/file/d/{uploaded['id']}/view")
        successes.append(day)
    except Exception as exc:
        print("  ERROR:",str(exc)[:550])
        failures.append((day,str(exc)[:250]))
    finally:
        for local in (WORK/f"source-day-{day:02d}.mp4", WORK/output_name):
            if local.exists(): local.unlink()

print("\n===== FINISHED =====")
print("Ready:",successes)
print("Failed:",failures)
print("Saved in YOUR Drive:",f"https://drive.google.com/drive/folders/{dest_id}")
print("Reply 'Done, videos compressed' in ChatGPT after this completes so the updated website can be rebuilt.")
